In [11]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML, IFrame

In [4]:
# 1. Load datasets
characters_df = pd.read_csv('data/characters.csv')
node_states_df = pd.read_csv('data/node_states.csv')
edges_df = pd.read_csv('data/temporal_edges.csv')

# Select the era you want to inspect interactively (e.g., 't11' for Revenge of the Sith)
target_era = 't11'

# 2. Initialize PyVis network canvas
net = Network(height='750px', width='100%', bgcolor='#1a1a1a', font_color='white')
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)

# 3. Filter states and edges for the chosen era
era_states = node_states_df[node_states_df['Era_Step'] == target_era].set_index('Node_ID')
era_edges = edges_df[edges_df['Era_Step'] == target_era]

In [14]:
# 4. Add nodes with advanced psychological styling (Size, Color, Borders, and Shapes)
for _, row in characters_df.iterrows():
    node_id = row['Node_ID']
    name = row['Character_Name']
    
    # Default baseline attributes
    color = '#4da6ff'
    border_color = '#2b78e4'
    border_width = 1
    node_size = 20
    node_shape = 'dot'
    title = f"<b>{name}</b><br>Faction: {row['Base_Faction']}"
    
    # Check if this node has active states in this era
    if node_id in era_states.index:
        state_info = era_states.loc[node_id]
        mental = state_info.get('Mental_State', 'Unknown')
        stress = float(state_info.get('Stress_Score', 0.0))
        prox = float(state_info.get('Palpatine_Proximity', 0.0))
        
        title += f"<br><b>Mental State:</b> {mental}"
        title += f"<br><b>Stress Score:</b> {stress:.2f}"
        title += f"<br><b>Palpatine Proximity:</b> {prox:.2f}"
        
        # Scale node size dynamically based on stress score
        node_size = 15 + (stress * 25)
        
        # Apply universal stress-based color grading across all characters
        if stress > 0.7:
            color = '#ff3333' # High stress (Bright Red)
        elif stress > 0.3:
            color = '#ff9933' # Moderate stress (Orange)
        else:
            color = '#4da6ff' # Low/Stable stress (Blue)
            
        # Highlight Palpatine Proximity with a thick gold border
        if prox > 0.5:
            border_color = '#ffd700'
            border_width = 4
            
        # Map node shape to qualitative mental state
        if mental in ['Fallen', 'Corrupted']:
            node_shape = 'box'
        elif mental in ['Conflicted', 'Unstable']:
            node_shape = 'triangle'
        else:
            node_shape = 'dot'

    net.add_node(
        node_id, 
        label=name, 
        title=title, 
        color={'background': color, 'border': border_color}, 
        borderWidth=border_width,
        size=node_size, 
        shape=node_shape
    )

# 5. Add edges for the snapshot
for _, row in era_edges.iterrows():
    source = row['Source']
    target = row['Target']
    weight = row.get('Weight', 1.0)
    net.add_edge(source, target, value=weight, title=f"Weight: {weight}")

In [15]:
# 6. Render and save the interactive HTML file
output_filename = f"anakin_network_{target_era}.html"
net.write_html(output_filename)

# Force Jupyter to render it properly inside a visible iframe box
display(IFrame(src=output_filename, width='100%', height='600px'))
print(f"Interactive network successfully rendered for era {target_era}.")

Interactive network successfully rendered for era t11.


In [16]:
import pandas as pd
from pyvis.network import Network
from IPython.display import display, IFrame
import time

# 1. Load datasets
characters_df = pd.read_csv('data/characters.csv')
node_states_df = pd.read_csv('data/node_states.csv')
edges_df = pd.read_csv('data/temporal_edges.csv')

# Select the era you want to inspect interactively
target_era = 't11'

# 2. Initialize PyVis network canvas
net = Network(height='750px', width='100%', bgcolor='#ffffff', font_color='black')
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=200)

# 3. Filter states and edges for the chosen era and index properly
era_states = node_states_df[node_states_df['Era_Step'] == target_era].set_index('Node_ID')
era_edges = edges_df[edges_df['Era_Step'] == target_era]

print(f"Found {len(era_states)} node states and {len(era_edges)} edges for era {target_era}.")

# 4. Add nodes with explicit psychological indicators
for _, row in characters_df.iterrows():
    node_id = row['Node_ID']
    name = row['Character_Name']
    
    # Defaults
    color = '#4da6ff'
    border_color = '#2b78e4'
    border_width = 1
    node_size = 20
    node_shape = 'dot'
    title = f"<b>{name}</b><br>Faction: {row.get('Base_Faction', 'Unknown')}"
    
    # Check if this node has active states in this era
    if node_id in era_states.index:
        state_info = era_states.loc[node_id]
        
        # Handle cases where pandas returns a Series or a DataFrame (if duplicate IDs exist)
        if isinstance(state_info, pd.DataFrame):
            state_info = state_info.iloc[0]
            
        mental = str(state_info.get('Mental_State', 'Unknown'))
        stress = float(state_info.get('Stress_Score', 0.0))
        prox = float(state_info.get('Palpatine_Proximity', 0.0))
        
        # Enhanced Tooltip
        title += f"<br><b>Mental State:</b> {mental}"
        title += f"<br><b>Stress Score:</b> {stress:.2f}"
        title += f"<br><b>Palpatine Proximity:</b> {prox:.2f}"
        
        # Dynamic Node Size scaling based on stress
        node_size = 15 + (stress * 30)
        
        # Dynamic Color grading based on stress thresholds
        if stress > 0.7:
            color = '#ff3333' # High stress (Bright Red)
        elif stress > 0.3:
            color = '#ff9933' # Moderate stress (Orange)
        else:
            color = '#4da6ff' # Low stress (Blue)
            
        # Palpatine Proximity Gold Border indicator
        if prox > 0.5:
            border_color = '#ffd700'
            border_width = 4
            
        # Shape mapping based on mental state
        if mental in ['Fallen', 'Corrupted']:
            node_shape = 'box'
        elif mental in ['Conflicted', 'Unstable']:
            node_shape = 'triangle'
        else:
            node_shape = 'dot'

    net.add_node(
        node_id, 
        label=name, 
        title=title, 
        color={'background': color, 'border': border_color}, 
        borderWidth=border_width,
        size=node_size, 
        shape=node_shape
    )

# 5. Add edges for the snapshot
for _, row in era_edges.iterrows():
    source = row['Source']
    target = row['Target']
    weight = float(row.get('Weight', 1.0))
    net.add_edge(source, target, value=weight, title=f"Weight: {weight}")

# 6. Render with a unique timestamp suffix to bypass browser cache
output_filename = f"anakin_network_{target_era}_{int(time.time())}.html"
net.write_html(output_filename)

display(IFrame(src=output_filename, width='100%', height='600px'))

Found 1 node states and 6 edges for era t11.
